In [3]:
import csv
import os
import sys
import shutil
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

for folder in ["work", "cancer"]:
    shutil.rmtree(folder, ignore_errors=True)

SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"
WORK = Path("work")
WORK.mkdir(exist_ok=True)


def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])


if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)

orders = pd.read_csv(DATA)
X = orders[["distance_km", "prep_time_min", "traffic_level", "rain"]]
y = orders["delivery_min"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_tr, y_tr)
print("Hardcoded MAE", round(mean_absolute_error(y_te, model.predict(X_te)), 2))

config_yaml = """data:
  path: ../data/delivery_times.csv
  features:
    - distance_km
    - prep_time_min
    - traffic_level
    - rain
  target: delivery_min

split:
  test_size: 0.2
  seed: 42
"""
(WORK / "config.yaml").write_text(config_yaml, encoding="utf-8")
print((WORK / "config.yaml").read_text(encoding="utf-8"))

with open(WORK / "config.yaml", encoding="utf-8") as fh:
    config = yaml.safe_load(fh)
print(config["data"]["features"])
print(config["split"]["test_size"])


def train_from_config(config, data_path):
    orders = pd.read_csv(data_path)
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"],
    )
    model = LinearRegression().fit(X_tr, y_tr)
    return model, mean_absolute_error(y_te, model.predict(X_te))


model, mae = train_from_config(config, DATA)
print("Config MAE", round(mae, 2))

for test_size in [0.2, 0.3, 0.4]:
    config["split"]["test_size"] = test_size
    _, mae = train_from_config(config, DATA)
    print("test_size", test_size, "-> MAE", round(mae, 2))
config["split"]["test_size"] = 0.2

assert 2 + 2 == 4
try:
    assert 2 + 2 == 5
except AssertionError:
    print("assert failed, as expected")

orders_py = """TRAFFIC_LEVELS = (1, 2, 3)

def minutes_per_km(delivery_min, distance_km):
    if distance_km <= 0:
        raise ValueError("distance_km must be positive")
    return delivery_min / distance_km

def is_valid_order(order):
    if order.get("distance_km", 0) <= 0:
        return False
    if order.get("prep_time_min", 0) < 0:
        return False
    if order.get("traffic_level") not in TRAFFIC_LEVELS:
        return False
    if order.get("rain") not in (0, 1):
        return False
    return True
"""
(WORK / "orders.py").write_text(orders_py, encoding="utf-8")

test_py = """import pytest
from orders import is_valid_order, minutes_per_km

def test_minutes_per_km_simple():
    assert minutes_per_km(30, 10) == 3.0

def test_minutes_per_km_rejects_zero_distance():
    with pytest.raises(ValueError):
        minutes_per_km(30, 0)

def test_good_order_is_accepted():
    good = {"distance_km": 5.0, "prep_time_min": 20, "traffic_level": 2, "rain": 0}
    assert is_valid_order(good) is True

def test_bad_traffic_level_is_rejected():
    bad = {"distance_km": 5.0, "prep_time_min": 20, "traffic_level": 9, "rain": 0}
    assert is_valid_order(bad) is False
"""
(WORK / "test_orders.py").write_text(test_py, encoding="utf-8")

ENV = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}


def run_pytest(*extra):
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", "-p", "no:cacheprovider", *extra],
        cwd=WORK, capture_output=True, text=True, env=ENV,
    )
    print(result.stdout)
    print("exit code:", result.returncode)
    return result


run = run_pytest()

original = (WORK / "orders.py").read_text(encoding="utf-8")
broken = original.replace("delivery_min / distance_km", "delivery_min * distance_km")
(WORK / "orders.py").write_text(broken, encoding="utf-8")
run_pytest("-x", "-q")
(WORK / "orders.py").write_text(original, encoding="utf-8")
print("module repaired")
print("broke successfully?", broken != original)

model_test_py = """import pandas as pd

def make_order(distance_km=5.0, prep_time_min=20, traffic_level=2, rain=0):
    return pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain,
    }])

def test_rain_never_makes_delivery_faster(trained_model):
    dry = trained_model.predict(make_order(rain=0))[0]
    wet = trained_model.predict(make_order(rain=1))[0]
    assert wet >= dry

def test_further_is_never_faster(trained_model):
    near = trained_model.predict(make_order(distance_km=2))[0]
    far = trained_model.predict(make_order(distance_km=12))[0]
    assert far > near

def test_error_is_better_than_guessing(model_mae, baseline_mae):
    assert model_mae < baseline_mae
"""
(WORK / "test_model.py").write_text(model_test_py, encoding="utf-8")

conftest_py = """import sys
from pathlib import Path
import pandas as pd
import pytest
import yaml
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE.parent))

@pytest.fixture
def config():
    with open(HERE / "config.yaml", encoding="utf-8") as fh:
        return yaml.safe_load(fh)

@pytest.fixture
def split(config):
    data_path = HERE.parent / "data" / "delivery_times.csv"
    orders = pd.read_csv(data_path)
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]
    return train_test_split(X, y, test_size=config["split"]["test_size"], random_state=config["split"]["seed"])

@pytest.fixture
def trained_model(split):
    X_tr, _, y_tr, _ = split
    return LinearRegression().fit(X_tr, y_tr)

@pytest.fixture
def model_mae(trained_model, split):
    _, X_te, _, y_te = split
    return mean_absolute_error(y_te, trained_model.predict(X_te))

@pytest.fixture
def baseline_mae(split):
    _, X_te, y_tr, y_te = split
    guess = [y_tr.mean()] * len(y_te)
    return mean_absolute_error(y_te, guess)
"""
(WORK / "conftest.py").write_text(conftest_py, encoding="utf-8")
run_pytest("-q")

config["training"] = {"max_rows": 300}
(WORK / "config.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
with open(WORK / "config.yaml", encoding="utf-8") as fh:
    config = yaml.safe_load(fh)
print(yaml.safe_dump(config, sort_keys=False))


def train_limited(config, data_path):
    orders = pd.read_csv(data_path).head(config["training"]["max_rows"])
    X = orders[config["data"]["features"]]
    y = orders[config["data"]["target"]]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"],
    )
    model = LinearRegression().fit(X_tr, y_tr)
    return mean_absolute_error(y_te, model.predict(X_te))


print("MAE on first", config["training"]["max_rows"], "orders:", round(train_limited(config, DATA), 2))

speed_test_py = """import pytest
from orders import minutes_per_km

def test_slow_delivery():
    assert minutes_per_km(60, 6) == 10.0

def test_negative_distance_raises():
    with pytest.raises(ValueError):
        minutes_per_km(30, -3)
"""
(WORK / "test_speed.py").write_text(speed_test_py, encoding="utf-8")
run_pytest("-q")

extra_test = """

def test_heavier_traffic_is_never_faster(trained_model):
    light = trained_model.predict(make_order(traffic_level=1))[0]
    heavy = trained_model.predict(make_order(traffic_level=3))[0]
    assert heavy >= light
"""
with open(WORK / "test_model.py", "a", encoding="utf-8") as fh:
    fh.write(extra_test)
run_pytest("-q")

CANCER = Path("cancer")
CANCER.mkdir(exist_ok=True)
(CANCER / "__init__.py").write_text("", encoding="utf-8")

cancer_config = {
    "data": {
        "features": [str(name) for name in load_breast_cancer().feature_names],
        "target": "target",
    },
    "split": {"test_size": 0.2, "seed": 42},
}
(CANCER / "config.yaml").write_text(yaml.safe_dump(cancer_config, sort_keys=False), encoding="utf-8")

cancer_model_py = """import math
from pathlib import Path
import joblib
import yaml
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent
CONFIG_PATH = HERE / "config.yaml"
MODEL_PATH = HERE / "model.joblib"

def load_config(path=CONFIG_PATH):
    with open(path, encoding="utf-8") as fh:
        return yaml.safe_load(fh)

def load_data(config):
    frame = load_breast_cancer(as_frame=True).frame
    return frame[config["data"]["features"]], frame[config["data"]["target"]]

def train_and_save_model(config=None, model_path=MODEL_PATH):
    config = config or load_config()
    X, y = load_data(config)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        random_state=config["split"]["seed"],
        stratify=y,
    )
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
    accuracy = accuracy_score(y_te, model.predict(X_te))
    joblib.dump(model, model_path)
    return model, accuracy

def feature_limits(config):
    X, _ = load_data(config)
    return (X.max() * 2).to_dict()

def is_valid_sample(sample, config=None):
    config = config or load_config()
    limits = feature_limits(config)
    for name in config["data"]["features"]:
        value = sample.get(name)
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            return False
        if not math.isfinite(value) or value < 0 or value > limits[name]:
            return False
    return True
"""
(CANCER / "model.py").write_text(cancer_model_py, encoding="utf-8")

sys.path.insert(0, str(Path.cwd()))
from cancer.model import train_and_save_model

_, cancer_accuracy = train_and_save_model()
print("Cancer model accuracy:", round(cancer_accuracy, 4))

validate_test_py = """from sklearn.datasets import load_breast_cancer
from cancer.model import is_valid_sample

def realistic_sample():
    row = load_breast_cancer(as_frame=True).data.iloc[0]
    return {name: float(value) for name, value in row.items()}

def test_realistic_sample_passes():
    assert is_valid_sample(realistic_sample()) is True

def test_unrealistic_sample_is_rejected():
    sample = realistic_sample()
    sample["mean radius"] = 500.0
    assert is_valid_sample(sample) is False
"""
(WORK / "test_validate.py").write_text(validate_test_py, encoding="utf-8")

cancer_test_py = """import pytest
from cancer.model import load_config, load_data, train_and_save_model

@pytest.fixture
def trained_model(tmp_path):
    model, _ = train_and_save_model(model_path=tmp_path / "model.joblib")
    return model

def test_same_sample_gives_same_prediction(trained_model):
    X, _ = load_data(load_config())
    sample = X.iloc[[0]]
    first = trained_model.predict(sample)[0]
    second = trained_model.predict(sample)[0]
    assert first == second
"""
(WORK / "test_cancer_model.py").write_text(cancer_test_py, encoding="utf-8")
run_pytest()

dataset ready: data/delivery_times.csv
Hardcoded MAE 1.92
data:
  path: ../data/delivery_times.csv
  features:
    - distance_km
    - prep_time_min
    - traffic_level
    - rain
  target: delivery_min

split:
  test_size: 0.2
  seed: 42

['distance_km', 'prep_time_min', 'traffic_level', 'rain']
0.2
Config MAE 1.92
test_size 0.2 -> MAE 1.92
test_size 0.3 -> MAE 2.01
test_size 0.4 -> MAE 2.04
assert failed, as expected
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
rootdir: /content/work
plugins: anyio-4.16.0, langsmith-0.14.17, typeguard-4.6.0, platformdirs-4.14.3
collecting ... collected 4 items

test_orders.py::test_minutes_per_km_simple PASSED                        [ 25%]
test_orders.py::test_minutes_per_km_rejects_zero_distance PASSED         [ 50%]
test_orders.py::test_good_order_is_accepted PASSED                       [ 75%]
test_orders.py::test_bad_traffic_level_

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-v', '-p', 'no:cacheprovider'], returncode=0, stdout='============================= test session starts ==============================\nplatform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3\nrootdir: /content/work\nplugins: anyio-4.16.0, langsmith-0.14.17, typeguard-4.6.0, platformdirs-4.14.3\ncollecting ... collected 13 items\n\ntest_cancer_model.py::test_same_sample_gives_same_prediction PASSED      [  7%]\ntest_model.py::test_rain_never_makes_delivery_faster PASSED              [ 15%]\ntest_model.py::test_further_is_never_faster PASSED                       [ 23%]\ntest_model.py::test_error_is_better_than_guessing PASSED                 [ 30%]\ntest_model.py::test_heavier_traffic_is_never_faster PASSED               [ 38%]\ntest_orders.py::test_minutes_per_km_simple PASSED                        [ 46%]\ntest_orders.py::test_minutes_per_km_rejects_zero_distance PASSED         [ 53%]\ntest_orders.py